In [32]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import torch
from torch.utils.data import Dataset, DataLoader

# Load the simplified GoEmotions dataset
print("Loading dataset splits...")
dataset = load_dataset("go_emotions", "simplified")

train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

Loading dataset splits...
Train size: 43410 | Val size: 5426 | Test size: 5427


In [33]:
# Transforming labels into a multi-hot encoded matrix
# Since each text can have multiple emotions (e.g., [0, 27]), we need a binary matrix for Scikit-Learn

print("Initializing MultiLabelBinarizer...")
mlb = MultiLabelBinarizer(classes=list(range(28)))

# Fit and transform the train labels, and only transform val/test
y_train = mlb.fit_transform(train_df['labels'])
y_val = mlb.transform(val_df['labels'])
y_test = mlb.transform(test_df['labels'])

print(f"Shape of y_train: {y_train.shape}")
print(f"Example of first row multi-hot label: {y_train[0]}")

Initializing MultiLabelBinarizer...
Shape of y_train: (43410, 28)
Example of first row multi-hot label: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]


In [34]:
# Converting text to numerical features using TF-IDF
# We use unigrams and bigrams, and limit features to 10,000 to manage memory

print("Fitting TF-IDF Vectorizer...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')

X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_val_tfidf = tfidf.transform(val_df['text'])
X_test_tfidf = tfidf.transform(test_df['text'])

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_val_tfidf: {X_val_tfidf.shape}")

Fitting TF-IDF Vectorizer...
Shape of X_train_tfidf: (43410, 10000)
Shape of X_val_tfidf: (5426, 10000)


In [35]:
# Training the Baseline Model
# We use Logistic Regression wrapped in OneVsRestClassifier for multilabel support
# class_weight='balanced' is crucial here due to the severe label imbalance found in Week 1

print("Training OneVsRest Baseline Model (Logistic Regression)...")
# Note: This might take a minute to run across 28 classes
baseline_model = OneVsRestClassifier(
    LogisticRegression(solver='lbfgs', max_iter=500, class_weight='balanced', n_jobs=-1)
)

baseline_model.fit(X_train_tfidf, y_train)
print("Training completed.")

Training OneVsRest Baseline Model (Logistic Regression)...
Training completed.


In [36]:
# Evaluating the Baseline Model on the Validation Set
print("Predicting on validation set...")
y_val_pred = baseline_model.predict(X_val_tfidf)

# Calculate Micro and Macro F1 Scores
micro_f1 = f1_score(y_val, y_val_pred, average='micro')
macro_f1 = f1_score(y_val, y_val_pred, average='macro')

print(f"Validation Micro F1-Score: {micro_f1:.4f}")
print(f"Validation Macro F1-Score: {macro_f1:.4f}\n")

# Print detailed classification report for a subset of classes to keep output clean
labels_names = dataset['train'].features['labels'].feature.names
print(classification_report(y_val, y_val_pred, target_names=labels_names, zero_division=0))

Predicting on validation set...
Validation Micro F1-Score: 0.4303
Validation Macro F1-Score: 0.3847

                precision    recall  f1-score   support

    admiration       0.49      0.81      0.61       488
     amusement       0.67      0.86      0.75       303
         anger       0.25      0.62      0.36       195
     annoyance       0.18      0.54      0.27       303
      approval       0.19      0.50      0.27       397
        caring       0.17      0.58      0.27       153
     confusion       0.14      0.55      0.22       152
     curiosity       0.11      0.42      0.17       248
        desire       0.23      0.69      0.34        77
disappointment       0.11      0.39      0.17       163
   disapproval       0.17      0.47      0.25       292
       disgust       0.23      0.63      0.34        97
 embarrassment       0.21      0.54      0.30        35
    excitement       0.14      0.52      0.22        96
          fear       0.42      0.69      0.52        90
  

In [37]:
# Creating Custom PyTorch Dataset class for Deep Learning phase
# This transitions our preprocessed data into PyTorch tensors for the neural network

class GoEmotionsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer # Will be injected in Week 3 (e.g., BERT tokenizer)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        # For baseline PyTorch MLP testing, we just return raw text and tensor labels
        # In Week 3, tokenizer logic will go here
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)

        return {
            'text': text,
            'labels': labels
        }

# Verify the Dataset works
print("Testing PyTorch Dataset instantiation...")
train_dataset = GoEmotionsDataset(train_df['text'].values, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

sample_batch = next(iter(train_loader))
print(f"Batch text size: {len(sample_batch['text'])}")
print(f"Batch labels shape: {sample_batch['labels'].shape}")

Testing PyTorch Dataset instantiation...
Batch text size: 16
Batch labels shape: torch.Size([16, 28])
